# 04 · Evaluate your final model

Run this after notebook 02, on the model saved in `outputs/best.pt`.
It answers three questions from the Week 10 slides:

| Step | Question |
|---|---|
| 4 | How good is the model at every threshold? (precision-recall and ROC curves) |
| 5–6 | Which threshold should the app use, given what each mistake costs? |
| 7 | How does the final model do on the test set at that threshold? |
| 8 | How did all your experiments compare? |

**Yes/no decisions.** A threshold makes sense for a yes/no decision:
"is this image the positive class or not?" With two classes, that is
your whole problem. With more classes, the notebook treats
`POSITIVE_CLASS` against all the others, which fits a project where
one class matters most (for example, "defective" among several kinds
of part).

**Definitions used below**

- **Threshold:** the model says "positive" when its probability for
  the positive class is at least the threshold. The default is 0.5.
- **False alarm** (false positive): the model says positive; the
  truth is not.
- **Missed case** (false negative): the truth is positive; the model
  says not.

## Step 1 · Setup
Same as notebooks 01 and 02. Use the same `REPO_URL`.

In [ ]:
# Run this cell first, every time you open the notebook.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# CHANGE: the address of YOUR team's copy of this repo on GitHub
REPO_URL = "https://github.com/dsc-scsu/csc451-project-starter.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content") / Path(REPO_URL).stem
    if not REPO_DIR.exists():
        !git clone -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    !git pull -q
elif Path.cwd().name == "notebooks":
    os.chdir("..")  # on your own computer: work from the repo folder

sys.path.insert(0, "src")
print("Working in", Path.cwd())

## Step 2 · Settings (CHANGE)

| Setting | What to put |
|---|---|
| `POSITIVE_CLASS` | The class that means "yes": the one you most need to find |
| `COST_MISSED` | How bad one missed case is |
| `COST_FALSE_ALARM` | How bad one false alarm is |
| `EVALUATE_TEST` | `True` once, when you have chosen your threshold |

Only the ratio of the two costs matters. `COST_MISSED = 5` and
`COST_FALSE_ALARM = 1` mean a missed case is 5 times worse than a
false alarm. Explain your choice of costs in your presentation.

In [ ]:
# CHANGE: settings for your project
PROJECT_FOLDER = "csc451_project"
DATA_ZIP_NAME = "data.zip"

POSITIVE_CLASS = "pneumonia"    # the class that means "yes"
COST_MISSED = 5                 # one missed case costs this much
COST_FALSE_ALARM = 1            # one false alarm costs this much

EVALUATE_TEST = False           # True once, at the very end

In [ ]:
# Where things are. You should not need to change this cell.
if IN_COLAB:
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    DATA_DIR = Path("/content/data")      # fast disk, erased after
else:                                     # the session ends
    PROJECT = Path("project_files")       # on your own computer
    DATA_DIR = PROJECT / "unzipped"
OUT = PROJECT / "outputs"                 # results to keep, on Drive
OUT.mkdir(parents=True, exist_ok=True)
print("Project folder:", PROJECT)

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (ConfusionMatrixDisplay, PrecisionRecallDisplay,
                             RocCurveDisplay, confusion_matrix)
from torch.utils.data import DataLoader

import data_utils as du
import train_utils as tu

device = "cuda" if torch.cuda.is_available() else "cpu"

## Step 3 · Load the final model and score the validation images
This rebuilds the model notebook 02 trained (from `model_info.json`), loads `best.pt`, and computes the probability of `POSITIVE_CLASS` for every validation image. Steps 4–6 use only the validation set.

In [ ]:
du.unzip_once(PROJECT / DATA_ZIP_NAME, DATA_DIR)
with open(OUT / "data_info.json") as f:
    IMAGE_ROOT = DATA_DIR / json.load(f)["image_root"]
with open(OUT / "classes.json") as f:
    classes = json.load(f)
with open(OUT / "model_info.json") as f:
    info = json.load(f)

if POSITIVE_CLASS not in classes:
    raise ValueError(f"POSITIVE_CLASS must be one of {classes}")
pos = classes.index(POSITIVE_CLASS)
if len(classes) == 2:
    negative_name = classes[1 - pos]
else:
    negative_name = "not " + POSITIVE_CLASS
    print(f"{len(classes)} classes: evaluating '{POSITIVE_CLASS}' "
          "against all the others")

model = tu.make_model(info["model"], len(classes), info["image_size"],
                      pretrained=False)
model.load_state_dict(torch.load(OUT / "best.pt", map_location=device))
model.to(device)
_, eval_tfms = tu.make_transforms(info["image_size"])

table = pd.read_csv(OUT / "splits.csv")

def positive_scores(split):
    """True yes/no labels and the model's P(positive) for one split."""
    rows = table[table["split"] == split]
    ds = du.ImageDataset(rows, IMAGE_ROOT, classes, eval_tfms)
    y, _, probs = tu.predict_all(model, DataLoader(ds, batch_size=64),
                                 device)
    return (y == pos).astype(int), probs[:, pos]

y_val, p_val = positive_scores("val")
print(f"{len(y_val)} validation images, {y_val.sum()} of them "
      f"'{POSITIVE_CLASS}'")

## Step 4 · Precision-recall and ROC curves

Each curve shows the model at **every** threshold at once, so you can
compare models without choosing a threshold first.

- **Precision-recall:** of the images the model calls positive, how
  many are (precision), against how many positives it finds (recall).
  The dashed line is a model that guesses. Useful when the positive
  class is rare.
- **ROC:** how many positives it finds against how many negatives it
  raises false alarms on.
- **AUC** (area under the curve) sums a curve up in one number: 1.0
  is perfect; 0.5 on the ROC curve is guessing.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
PrecisionRecallDisplay.from_predictions(
    y_val, p_val, ax=ax1, name=info["model"], plot_chance_level=True)
ax1.set_title(f"Precision-recall: '{POSITIVE_CLASS}'")
ax1.set_xlabel("recall: share of positives found")
ax1.set_ylabel("precision: share of 'yes' answers that are right")
RocCurveDisplay.from_predictions(
    y_val, p_val, ax=ax2, name=info["model"], plot_chance_level=True)
ax2.set_title(f"ROC: '{POSITIVE_CLASS}'")
ax2.set_xlabel("false positive rate: share of negatives flagged")
ax2.set_ylabel("true positive rate (recall)")
plt.tight_layout()
plt.savefig(OUT / "pr_roc_curves.png", dpi=150)
plt.show()

## Step 5 · Choose the threshold from the cost of each mistake

For every threshold from 0.01 to 0.99, count the missed cases and
false alarms on the validation set and add up their cost. The cheapest
threshold is the one to use. Put it in your app and your report.

In [ ]:
def count_mistakes(y, p, threshold):
    said_yes = p >= threshold
    missed = int(((y == 1) & ~said_yes).sum())
    false_alarms = int(((y == 0) & said_yes).sum())
    return missed, false_alarms

def total_cost(y, p, threshold):
    missed, false_alarms = count_mistakes(y, p, threshold)
    return COST_MISSED * missed + COST_FALSE_ALARM * false_alarms

thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
costs = [total_cost(y_val, p_val, t) for t in thresholds]
# The cheapest threshold; if several tie, the one closest to 0.5
cheapest = thresholds[np.array(costs) == min(costs)]
best_t = cheapest[np.argmin(np.abs(cheapest - 0.5))]

plt.figure(figsize=(8, 4))
plt.plot(thresholds, costs)
plt.axvline(0.5, color="gray", linestyle="--", label="default 0.5")
plt.axvline(best_t, color="red", label=f"cheapest {best_t:.2f}")
plt.xlabel("threshold")
plt.ylabel("total cost on validation")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "cost_vs_threshold.png", dpi=150)
plt.show()

for t in (0.5, best_t):
    missed, false_alarms = count_mistakes(y_val, p_val, t)
    print(f"threshold {t:.2f}: {missed} missed, {false_alarms} false "
          f"alarms, cost {total_cost(y_val, p_val, t)}")

## Step 6 · What moving the threshold changes
The same validation images at the default threshold and at the chosen one. Moving the threshold trades one kind of mistake for the other; it does not make the model better.

In [ ]:
names = [negative_name, POSITIVE_CLASS]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, t in zip(axes, (0.5, best_t)):
    cm = confusion_matrix(y_val, (p_val >= t).astype(int), labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=names).plot(
        ax=ax, colorbar=False)
    ax.set_title(f"Validation, threshold {t:.2f}")
plt.tight_layout()
plt.show()

## Step 7 · Test set (once, at the end)

Set `EVALUATE_TEST = True` only after you have chosen the model and
the threshold. The threshold comes from the validation set; the test
set only reports how well that choice works on new images.

In [ ]:
if not EVALUATE_TEST:
    print("Skipped. Set EVALUATE_TEST = True when you have chosen.")
else:
    y_test, p_test = positive_scores("test")
    for t in (0.5, best_t):
        missed, false_alarms = count_mistakes(y_test, p_test, t)
        acc = ((p_test >= t).astype(int) == y_test).mean()
        print(f"Test, threshold {t:.2f}: accuracy {acc:.1%}, "
              f"{missed} missed, {false_alarms} false alarms, "
              f"cost {total_cost(y_test, p_test, t)}")
    cm = confusion_matrix(y_test, (p_test >= best_t).astype(int),
                          labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=names).plot(colorbar=False)
    plt.title(f"Test set, threshold {best_t:.2f}")
    plt.tight_layout()
    plt.savefig(OUT / "test_confusion_matrix.png", dpi=150)
    plt.show()

## Step 8 · Compare your experiments
Every run of notebook 02 added a row to `outputs/experiments.csv`. This chart shows them side by side, with the largest-class baseline, for your presentation. Runs are labeled with their `RUN_NOTE`, so write useful notes.

In [ ]:
log = pd.read_csv(OUT / "experiments.csv")
labels = [f"{n}  [{m}]" for n, m in zip(log["note"], log["model"])]

plt.figure(figsize=(9, 0.45 * len(log) + 1.5))
plt.barh(labels, log["best_val_acc"])
plt.axvline(log["baseline_val"].iloc[-1], color="gray", linestyle="--",
            label="largest-class baseline")
plt.xlabel("best validation accuracy")
plt.xlim(0, 1)
plt.gca().invert_yaxis()   # first run at the top
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUT / "experiments_chart.png", dpi=150)
plt.show()

## Done

Saved in `outputs/`: `pr_roc_curves.png`, `cost_vs_threshold.png`,
`experiments_chart.png`, and `test_confusion_matrix.png` once you run
the test step.

**Using the threshold in the app.** For a two-class model, change the
prediction in `app/app.py` from `probs.argmax()` to a comparison with
your threshold:

```python
THRESHOLD = 0.30                       # from Step 5
pos = classes.index("pneumonia")       # your POSITIVE_CLASS
best = pos if probs[pos] >= THRESHOLD else 1 - pos
```